# GGE: Generated Genetic Expression Evaluator

This notebook demonstrates the core functionality of the `gge-eval` library for evaluating gene expression generative models.

**Paper**: A Standardized Framework for Evaluating Gene Expression Generative Models  
**Accepted at**: Gen2 Workshop at ICLR 2026

## Setup

Install the library if needed:
```bash
pip install gge-eval
```

In [1]:
import sys
sys.path.insert(0, '../src')

In [2]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

# GGE imports
from gge import (
    evaluate,
    evaluate_lazy,
    evaluate_deg_space,
    evaluate_pc_space,
    compute_perturbation_effect_correlation,
)
from gge.metrics import (
    PearsonCorrelation,
    SpearmanCorrelation,
    RSquared,
    Wasserstein1Distance,
    Wasserstein2Distance,
    MMDDistance,
    MSEDistance,
)

print("GGE imported successfully!")


[KeOps] Warning : There were warnings or errors :
/bin/sh: brew: command not found

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.

[KeOps] Warning : There were warnings or errors :
/bin/sh: brew: command not found

[KeOps] Warning : OpenMP library not found, it must be downloaded through Homebrew for apple Silicon chips
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.
GGE imported successfully!


## Create Synthetic Gene Expression Data

We create realistic synthetic data with:
- 200 samples (100 control, 100 perturbed)
- 500 genes
- Generated data that approximates real data with some noise

In [3]:
np.random.seed(42)

n_samples = 200
n_genes = 500
n_control = 100
n_perturbed = 100

# Generate base expression (log-normal distribution typical of RNA-seq)
base_expression = np.random.lognormal(mean=2, sigma=1, size=(n_samples, n_genes))

# Add perturbation effect to second half (perturbed samples)
# Some genes are differentially expressed (DEGs)
n_degs = 50
deg_indices = np.random.choice(n_genes, n_degs, replace=False)
perturbation_effect = np.zeros(n_genes)
perturbation_effect[deg_indices] = np.random.uniform(-2, 2, n_degs)  # Log2 fold change

# Apply perturbation effect
real_expression = base_expression.copy()
real_expression[n_control:, :] *= np.exp(perturbation_effect)  # Apply fold change

# Create "generated" data - approximates real with some noise
generated_expression = real_expression + np.random.normal(0, 0.5, real_expression.shape)
generated_expression = np.maximum(generated_expression, 0)  # Non-negative

# Create observation metadata
obs_data = pd.DataFrame({
    'condition': ['control'] * n_control + ['perturbed'] * n_perturbed,
    'cell_type': np.random.choice(['TypeA', 'TypeB'], n_samples),
    'split': np.random.choice(['train', 'test'], n_samples, p=[0.7, 0.3]),
})

# Create gene metadata
var_data = pd.DataFrame({
    'gene_name': [f'Gene_{i}' for i in range(n_genes)],
}, index=[f'Gene_{i}' for i in range(n_genes)])

# Create AnnData objects
real_adata = ad.AnnData(
    X=real_expression,
    obs=obs_data.copy(),
    var=var_data.copy(),
)

generated_adata = ad.AnnData(
    X=generated_expression,
    obs=obs_data.copy(),
    var=var_data.copy(),
)

print(f"Real data: {real_adata.shape}")
print(f"Generated data: {generated_adata.shape}")
print(f"Conditions: {real_adata.obs['condition'].value_counts().to_dict()}")
print(f"Number of DEGs: {n_degs}")

Real data: (200, 500)
Generated data: (200, 500)
Conditions: {'control': 100, 'perturbed': 100}
Number of DEGs: 50


/Users/ar36/miniforge3/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/Users/ar36/miniforge3/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


## Basic Evaluation

The `evaluate()` function computes all metrics across conditions.

In [4]:
# Run evaluation
results = evaluate(
    real_data=real_adata,
    generated_data=generated_adata,
    condition_columns=['condition'],
    metrics=['pearson', 'spearman', 'r_squared', 'wasserstein_1', 'mmd'],
)

# Display summary
print(results.summary())

Evaluating 1 splits: ['all']
Using 7 metrics: ['pearson', 'spearman', 'r_squared', 'wasserstein_1', 'mmd', 'multivariate_wasserstein', 'multivariate_mmd']

  Split 'all': 2 conditions

EVALUATION SUMMARY

Split: all (2 conditions)
----------------------------------------
  mmd: 0.0000 ± 0.0000
  multivariate_mmd: 0.0000 ± 0.0000
  multivariate_wasserstein: 62.3453 ± 0.3058
  pearson: 0.9991 ± 0.0002
  r_squared: 0.9995 ± 0.0004
  spearman: 0.9927 ± 0.0010
  wasserstein_1: 0.2358 ± 0.0010
{'n_splits': 1, 'n_genes': 500, 'condition_columns': ['condition'], 'splits': {'all': {'n_conditions': 2, 'aggregates': {'pearson_mean': 0.9991409826235983, 'pearson_std': 0.00015734991331151083, 'pearson_median': 0.9991409826235983, 'multivariate_wasserstein_mean': 62.34530067443848, 'multivariate_wasserstein_std': 0.3057727813720703, 'multivariate_wasserstein_median': 62.34530067443848, 'r_squared_mean': 0.9995293026776938, 'r_squared_std': 0.0004422009666581461, 'r_squared_median': 0.999529302677693

## Individual Metrics

You can also compute metrics individually for more control.

In [5]:
# Compute individual metrics
pearson = PearsonCorrelation()
spearman = SpearmanCorrelation()
r_squared = RSquared()
mse = MSEDistance()

# Get data for perturbed condition
mask = real_adata.obs['condition'] == 'perturbed'
real_perturbed = real_adata[mask].X
gen_perturbed = generated_adata[mask].X

# Compute metrics
pearson_result = pearson.compute(real_perturbed, gen_perturbed)
spearman_result = spearman.compute(real_perturbed, gen_perturbed)
r2_result = r_squared.compute(real_perturbed, gen_perturbed)
mse_result = mse.compute(real_perturbed, gen_perturbed)

print("Perturbed condition metrics:")
print(f"  Pearson correlation: {pearson_result.aggregate_value:.4f}")
print(f"  Spearman correlation: {spearman_result.aggregate_value:.4f}")
print(f"  R-squared: {r2_result.aggregate_value:.4f}")
print(f"  MSE: {mse_result.aggregate_value:.4f}")

Perturbed condition metrics:
  Pearson correlation: 0.9990
  Spearman correlation: 0.9917
  R-squared: 1.0000
  MSE: 0.0024


## Perturbation-Effect Correlation (Paper Eq. 1)

This metric measures whether the model captures the **direction and magnitude** of perturbation effects:

$$\rho_{effect} = \text{corr}(\mu_{real} - \mu_{ctrl}, \mu_{gen} - \mu_{ctrl})$$

This is more informative than raw correlation when evaluating perturbation models.

In [6]:
# Compute control mean
control_mask = real_adata.obs['condition'] == 'control'
control_mean = real_adata[control_mask].X.mean(axis=0)

# Get perturbed samples
perturbed_mask = real_adata.obs['condition'] == 'perturbed'

# Compute perturbation-effect correlation
rho_effect = compute_perturbation_effect_correlation(
    real_perturbed=real_adata[perturbed_mask].X,
    generated_perturbed=generated_adata[perturbed_mask].X,
    control_mean=control_mean,
    method='pearson',
)

print(f"Perturbation-effect correlation: {rho_effect:.4f}")
print("\nInterpretation: Higher values indicate the model better captures")
print("the direction and magnitude of perturbation effects.")

Perturbation-effect correlation: 1.0000

Interpretation: Higher values indicate the model better captures
the direction and magnitude of perturbation effects.


## PC-Space Evaluation

For global structure comparison, evaluate in principal component space.

In [7]:
# Evaluate in PC space (50 principal components)
pc_results = evaluate_pc_space(
    real_data=real_adata,
    generated_data=generated_adata,
    condition_columns=['condition'],
    n_components=50,
)

print("PC-Space Evaluation Results:")
print(pc_results.summary())

Transforming data to 50-component PC space...


/Users/ar36/Desktop/GenEval/geneval/examples/../src/gge/utils/pca.py:263: UserWarning: Could not project to reference PC space. Computing PCA separately on generated data.
  warnings.warn(


Evaluating with metrics: ['wasserstein_1', 'wasserstein_2', 'mmd', 'energy']
Evaluating 1 splits: ['all']
Using 6 metrics: ['wasserstein_1', 'wasserstein_2', 'mmd', 'energy', 'multivariate_wasserstein', 'multivariate_mmd']

  Split 'all': 2 conditions

EVALUATION SUMMARY

Split: all (2 conditions)
----------------------------------------
  energy: 0.0574 ± 0.0009
  mmd: 0.0000 ± 0.0000
  multivariate_mmd: 0.0000 ± 0.0000
  multivariate_wasserstein: 10491.1729 ± 68.3867
  wasserstein_1: 2.0463 ± 0.0357
  wasserstein_2: 2.3382 ± 0.3240
PC-Space Evaluation Results:
{'n_splits': 1, 'n_genes': 50, 'condition_columns': ['condition'], 'splits': {'all': {'n_conditions': 2, 'aggregates': {'multivariate_wasserstein_mean': 10491.1728515625, 'multivariate_wasserstein_std': 68.38671875, 'multivariate_wasserstein_median': 10491.1728515625, 'multivariate_mmd_mean': 0.0, 'multivariate_mmd_std': 0.0, 'multivariate_mmd_median': 0.0, 'mmd_mean': 3.1486352243259573e-07, 'mmd_std': 3.1486352243259573e-07, 

## Mixed-Space Evaluation (Paper API)

The `evaluate_lazy()` function allows metrics to be computed in different spaces - raw, PCA, or DEG space.

Each metric can specify its own computation space:
- `space="raw"`: Original gene expression space (default)
- `space="pca"`: Principal component space with `n_components` dimensions
- `space="deg"`: Differentially expressed genes space with `deg_lfc` and `deg_pval` thresholds

In [8]:
# Define metrics with different spaces
metrics = [
    # Correlation in raw space
    PearsonCorrelation(space="raw"),
    SpearmanCorrelation(space="raw"),
    
    # Distances in PCA space (50 components)
    Wasserstein2Distance(space="pca", n_components=50),
    MMDDistance(space="pca", n_components=50),
    
    # R-squared in DEG space
    RSquared(space="deg", deg_lfc=0.5, deg_pval=0.05),
]

# Check metric names (automatically include space suffix)
for m in metrics:
    print(f"{m.__class__.__name__}: name='{m.name}', space='{m.space}'")

PearsonCorrelation: name='pearson', space='raw'
SpearmanCorrelation: name='spearman', space='raw'
Wasserstein2Distance: name='wasserstein_2_pca50', space='pca'
MMDDistance: name='mmd_pca50', space='pca'
RSquared: name='r_squared_deg', space='deg'


## Summary

This notebook demonstrated:

1. **Basic evaluation** with `evaluate()` function
2. **Individual metrics** (Pearson, Spearman, R-squared, MSE)
3. **Perturbation-effect correlation** (Paper Eq. 1)
4. **PC-space evaluation** for global structure
5. **Mixed-space API** with `evaluate_lazy()` and per-metric space configuration

For more advanced usage, see the documentation at https://andrearubbi.github.io/GGE/